In [1]:
import sys
sys.path.append('../')

import pandas as pd
from functools import partial
from util.polymarket_client import PolymarketAPIClient
from trading_rules import entries as e, exits as ex, engine
from trading_rules.metrics import Metrics
from util.data_processor import TickDataIntervalEnum

In [2]:
# FETCH THE DATA STEP AND PROCESS IT
MARKET_SLUG = 'will-jesus-christ-return-in-2025'
client = PolymarketAPIClient()
market = client.get_price_history_by_outcome(MARKET_SLUG, desired_outcome="No", interval=TickDataIntervalEnum.FIVE_MINUTES)
market["symbol"] = MARKET_SLUG
resampled_df = market.resample("10min").last()

Will Jesus Christ return in 2025?
Not Will Jesus Christ return in 2025?
requested start and end: 2024-12-31 12:00:00+00:00 2025-12-31 12:00:00+00:00


In [3]:
resampled_df

,timestamp,open,high,low,close,volume,outcome_id,symbol
timestamp_formatted,,,,,,,,
2025-03-20 22:10:00+00:00,1.742509e+12,0.9700,0.9700,0.9700,0.9700,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
2025-03-20 22:20:00+00:00,1.742510e+12,0.9695,0.9695,0.9695,0.9695,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
2025-03-20 22:30:00+00:00,1.742510e+12,0.9650,0.9650,0.9650,0.9650,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
2025-03-20 22:40:00+00:00,1.742511e+12,0.9670,0.9670,0.9670,0.9670,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
2025-03-20 22:50:00+00:00,1.742511e+12,0.9610,0.9610,0.9610,0.9610,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
...,...,...,...,...,...,...,...,...
2025-12-31 11:10:00+00:00,1.767180e+12,0.9995,0.9995,0.9995,0.9995,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
2025-12-31 11:20:00+00:00,1.767180e+12,0.9995,0.9995,0.9995,0.9995,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025
2025-12-31 11:30:00+00:00,1.767181e+12,0.9995,0.9995,0.9995,0.9995,0.0,4334699957357127908482970555095165722856012218...,will-jesus-christ-return-in-2025


In [4]:
# RUN THE SIMULATED BACKTEST

entries_partials = [
    partial(e.mean_reversion, data=resampled_df, window=10)
]
entries_series = [entry(data=resampled_df) for entry in entries_partials]
# AND all the entry rules
entries = pd.concat(entries_series, axis=1).all(axis=1)
exit_rules = [ex.time_exit(max_bars=144)]

result = engine.backtest(
    data=resampled_df,
    entries_series=entries,
    exits_list=exit_rules,
    initial_cash=10000,
)

result


{'equity': timestamp_formatted
 2025-03-20 22:10:00+00:00    10000.000000
 2025-03-20 22:20:00+00:00    10000.000000
 2025-03-20 22:30:00+00:00    10000.000000
 2025-03-20 22:40:00+00:00    10000.000000
 2025-03-20 22:50:00+00:00    10000.000000
                                  ...     
 2025-12-31 11:10:00+00:00    10881.900953
 2025-12-31 11:20:00+00:00    10881.900953
 2025-12-31 11:30:00+00:00    10881.900953
 2025-12-31 11:40:00+00:00    10881.900953
 2025-12-31 11:50:00+00:00    10881.900953
 Freq: 10min, Length: 41123, dtype: float64,
 'trades': [{'entry_price': 0.9625,
   'qty': 10389.610389610389,
   'entry_idx': 9,
   'exit_price': 0.971,
   'exit_idx': 153},
  {'entry_price': 0.971,
   'qty': 10389.610389610389,
   'entry_idx': 157,
   'exit_price': 0.9685,
   'exit_idx': 301},
  {'entry_price': 0.9685,
   'qty': 10389.610389610389,
   'entry_idx': 302,
   'exit_price': 0.969,
   'exit_idx': 446},
  {'entry_price': 0.9685,
   'qty': 10394.974153363413,
   'entry_idx': 459,


In [6]:
# MEASURE PERFORMANCE USING METRICS
metrics = Metrics(result['trades'], result['equity'])
metrics.summary()

,Total Return,CAR,Sharpe,Max Drawdown,Trades
0,0.08819,0.114157,3.265826,-0.019659,189
